In [0]:
# ============================================
# NOTEBOOK 2: Silver Layer - Data Cleaning
# PROJECT: Wikipedia Clickstream Analytics
# ============================================

# Load bronze tables
bronze_clickstream = spark.table("workspace.default.bronze_clickstream")
bronze_pageviews = spark.table("workspace.default.bronze_pageviews")

print("Bronze Clickstream count:", bronze_clickstream.count())
print("Bronze Pageviews count:", bronze_pageviews.count())
print("✅ Bronze tables loaded successfully!")

Bronze Clickstream count: 34170973
Bronze Pageviews count: 4977578
✅ Bronze tables loaded successfully!


In [0]:
# ============================================
# SILVER: Clean Clickstream Data
# ============================================
from pyspark.sql.functions import col, current_timestamp

# 1. Remove bot traffic (other-empty, other-internal are bots/automated)
# 2. Keep only valid click types
# 3. Remove nulls
# 4. Remove zero or negative click counts

valid_click_types = ["link", "external", "other"]

silver_clickstream = bronze_clickstream \
    .filter(col("source_page").isNotNull()) \
    .filter(col("target_page").isNotNull()) \
    .filter(col("click_type").isin(valid_click_types)) \
    .filter(col("click_count") > 0) \
    .filter(~col("source_page").startswith("other-")) \
    .withColumn("ingested_at", current_timestamp())

print("Bronze clickstream count:", bronze_clickstream.count())
print("Silver clickstream count:", silver_clickstream.count())
print("Records removed:", bronze_clickstream.count() - silver_clickstream.count())
display(silver_clickstream.limit(5))

Bronze clickstream count: 34170973
Silver clickstream count: 24522123
Records removed: 9648850


source_page,target_page,click_type,click_count,ingested_at
Chinese_water_dragon,Arthropod,link,12,2026-04-24T05:55:21.059Z
Chinese_water_snake,Snake,link,12,2026-04-24T05:55:21.059Z
Chinese_water_torture,Waterboarding,other,12,2026-04-24T05:55:21.059Z
Chinese_zodiac,Shang_dynasty,link,12,2026-04-24T05:55:21.059Z
Chinese_zodiac,Wild_boar,link,12,2026-04-24T05:55:21.059Z


In [0]:
# ============================================
# SILVER: Timestamp Normalization
# ============================================
from pyspark.sql.functions import col, date_trunc, hour

# First reload silver tables from Delta
silver_clickstream = spark.table("workspace.default.silver_clickstream")
silver_pageviews = spark.table("workspace.default.silver_pageviews")

print("✅ Silver tables reloaded!")
print("Clickstream count:", silver_clickstream.count())

# Add normalized hour bucket timestamp to clickstream
silver_clickstream = silver_clickstream \
    .withColumn("hour_bucket", 
        date_trunc("hour", col("ingested_at"))) \
    .withColumn("hour_of_day", 
        hour(col("ingested_at")))

# Add normalized hour bucket to pageviews
silver_pageviews = silver_pageviews \
    .withColumn("hour_bucket", 
        date_trunc("hour", col("ingested_at"))) \
    .withColumn("hour_of_day", 
        hour(col("ingested_at")))

print("✅ Timestamps normalized to UTC hourly buckets!")
display(silver_clickstream.select(
    "source_page", "target_page", 
    "ingested_at", "hour_bucket", "hour_of_day"
).limit(5))

✅ Silver tables reloaded!
Clickstream count: 24522123
✅ Timestamps normalized to UTC hourly buckets!


source_page,target_page,ingested_at,hour_bucket,hour_of_day
¡Ah_qué_Kiko!,El_Chavo_del_Ocho,2026-04-25T07:59:08.283Z,2026-04-25T07:00:00.000Z,7
¡Ay_Carmela!_(song),Battle_of_the_Ebro,2026-04-25T07:59:08.283Z,2026-04-25T07:00:00.000Z,7
¡Mayday!,Believers_(¡Mayday!_album),2026-04-25T07:59:08.283Z,2026-04-25T07:00:00.000Z,7
¡Mayday!,Stuck_on_an_Island,2026-04-25T07:59:08.283Z,2026-04-25T07:00:00.000Z,7
¡Mucha_Lucha!,List_of_programs_broadcast_by_Cartoon_Network,2026-04-25T07:59:08.283Z,2026-04-25T07:00:00.000Z,7


In [0]:
# ============================================
# SILVER: Clean Pageview Data
# ============================================
from pyspark.sql.functions import col, current_timestamp, regexp_replace

# 1. Keep only English Wikipedia (language_code = 'en')
# 2. Remove special pages (File:, Module:, Special:, Talk:, User:)
# 3. Remove null article titles
# 4. Remove zero view counts

silver_pageviews = bronze_pageviews \
    .filter(col("language_code") == "en") \
    .filter(col("article_title").isNotNull()) \
    .filter(col("view_count") > 0) \
    .filter(~col("article_title").startswith("File:")) \
    .filter(~col("article_title").startswith("Special:")) \
    .filter(~col("article_title").startswith("Module:")) \
    .filter(~col("article_title").startswith("Talk:")) \
    .filter(~col("article_title").startswith("User:")) \
    .withColumn("ingested_at", current_timestamp())

print("Bronze pageviews count:", bronze_pageviews.count())
print("Silver pageviews count:", silver_pageviews.count())
print("Records removed:", bronze_pageviews.count() - silver_pageviews.count())
display(silver_pageviews.limit(5))

Bronze pageviews count: 4977578
Silver pageviews count: 861793
Records removed: 4115785


language_code,article_title,view_count,response_size,ingested_at
en,!Bang!,1,0,2026-04-24T05:57:13.439Z
en,!Kung_languages,1,0,2026-04-24T05:57:13.439Z
en,!Kung_mythology,1,0,2026-04-24T05:57:13.439Z
en,!Lander,1,0,2026-04-24T05:57:13.439Z
en,!_(Trippie_Redd_album),2,0,2026-04-24T05:57:13.439Z


In [0]:
# ============================================
# SILVER: Save Cleaned Data to Delta Tables (Fixed)
# ============================================

# Save silver clickstream with schema overwrite
silver_clickstream.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.silver_clickstream")

print("✅ silver_clickstream table saved!")

# Save silver pageviews with schema overwrite
silver_pageviews.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.silver_pageviews")

print("✅ silver_pageviews table saved!")
print("🎉 Silver Layer Complete!")

✅ silver_clickstream table saved!
✅ silver_pageviews table saved!
🎉 Silver Layer Complete!


In [0]:
# ============================================
# SILVER: Unit Tests — Validate Transformations
# ============================================
print("Running unit tests...")

silver_clickstream = spark.table("workspace.default.silver_clickstream")
silver_pageviews = spark.table("workspace.default.silver_pageviews")

# Test 1: No null source pages
null_sources = silver_clickstream.filter(
    silver_clickstream.source_page.isNull()).count()
assert null_sources == 0, f"FAIL: {null_sources} null source pages found!"
print("✅ Test 1 Passed: No null source pages")

# Test 2: No null target pages
null_targets = silver_clickstream.filter(
    silver_clickstream.target_page.isNull()).count()
assert null_targets == 0, f"FAIL: {null_targets} null target pages found!"
print("✅ Test 2 Passed: No null target pages")

# Test 3: Only valid click types
invalid_types = silver_clickstream.filter(
    ~silver_clickstream.click_type.isin(["link", "other"])
).count()
assert invalid_types == 0, f"FAIL: {invalid_types} invalid click types!"
print("✅ Test 3 Passed: All click types valid")

# Test 4: No bot traffic (other- prefixed pages)
bot_pages = silver_clickstream.filter(
    silver_clickstream.source_page.startswith("other-")).count()
assert bot_pages == 0, f"FAIL: {bot_pages} bot pages found!"
print("✅ Test 4 Passed: No bot traffic")

# Test 5: Only English pageviews
non_english = silver_pageviews.filter(
    silver_pageviews.language_code != "en").count()
assert non_english == 0, f"FAIL: {non_english} non-English pages!"
print("✅ Test 5 Passed: Only English pageviews")

# Test 6: No zero or negative view counts
bad_views = silver_pageviews.filter(
    silver_pageviews.view_count <= 0).count()
assert bad_views == 0, f"FAIL: {bad_views} zero/negative views!"
print("✅ Test 6 Passed: All view counts positive")

# Test 7: Record count check
cs_count = silver_clickstream.count()
pv_count = silver_pageviews.count()
assert cs_count > 10000000, f"FAIL: Only {cs_count} clickstream records!"
assert pv_count > 100000, f"FAIL: Only {pv_count} pageview records!"
print(f"✅ Test 7 Passed: {cs_count:,} clickstream + {pv_count:,} pageview records")

print("\n🎉 ALL 7 UNIT TESTS PASSED!")

Running unit tests...
✅ Test 1 Passed: No null source pages
✅ Test 2 Passed: No null target pages
✅ Test 3 Passed: All click types valid
✅ Test 4 Passed: No bot traffic
✅ Test 5 Passed: Only English pageviews
✅ Test 6 Passed: All view counts positive
✅ Test 7 Passed: 24,522,123 clickstream + 861,793 pageview records

🎉 ALL 7 UNIT TESTS PASSED!
